# 06m2 — Generalization: 50Salads (action-segmentation family, Phase 2)

**Role: single-group representation quality.** 50 videos, all one activity (`salad`), so there is
**no across-class order-null** (the null needs >=2 groups) — the order table is empty by construction.
The deliverable is the **prototypical salad execution**: the barycenter of all 50 executions, and how
faithfully each averager recovers it.

**Caveat (config-vs-real):** 50Salads' `mapping.txt` has two boundary markers, `action_start` (id 17)
and `action_end` (id 18). `read_mapping` remaps only the first present (`action_start`->0); `action_end`
stays an ordinary symbol. Non-blocking, but noted for honesty. Kernel: `smartflat_repro`.

In [1]:
%load_ext autoreload
%autoreload 2
import os
os.environ.setdefault('NUMBA_THREADING_LAYER', 'workqueue')  # fork-safe rTWE under nbconvert
os.environ.setdefault('MPLBACKEND', 'agg')                   # headless figures
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from collections import Counter
from IPython.display import display

from smartflat.utils.utils_io import get_data_root
from smartflat.utils.utils import upsample_sequence
from smartflat.features.symbolic_barycenter.generalization.action_segmentation import (
    download_action_seg, load_action_seg, build_action_seg_ground_cost,
    default_root, read_mapping, DATASETS)
from smartflat.features.symbolic_barycenter.generalization.suite import run_generalization_suite
from smartflat.features.symbolic_barycenter.registries import default_baseline_methods
from smartflat.features.symbolic_barycenter.visualization import plot_cohort_barycenters

NAME, UPSAMPLE = '50salads', 32   # UPSAMPLE ~= 2x median segment count (median 19); short procedural sequences
OUT = os.path.join(get_data_root(), 'outputs', 'symbolic_barycenter', 'generalization', NAME)
os.makedirs(OUT, exist_ok=True)
print('dataset:', NAME, '| upsample_to:', UPSAMPLE, '| output dir:', OUT)

dataset: 50salads | upsample_to: 32 | output dir: /home/perochon/data-gold-final/outputs/symbolic_barycenter/generalization/50salads


## 1. Load + config-vs-real-files sanity

In [2]:
# GT action labels used DIRECTLY as symbols (RLE'd -> segment sequences, background -> 0).
# download_action_seg pulls only the tiny GT text via HTTP Range (never the 30 GB features);
# it is flag-guarded, so this is a no-op once the data is cached.
download_action_seg(NAME)
meta, X, labels, G = load_action_seg(NAME)
cfg = DATASETS[NAME]
ns = meta['n_segments'].to_numpy()
display(pd.DataFrame({
    'metric': ['N videos (loaded)', '|V| = G (incl. background 0)', '#activity classes',
               'segment length p10/50/90', 'published N', 'published #classes'],
    'value':  [len(X), G, len(set(labels)),
               tuple(np.percentile(ns, [10, 50, 90]).round(1)),
               cfg['n_videos'], cfg['n_classes']],
}))
print('activities:', sorted(set(labels)))
print('per-activity video counts:', dict(Counter(labels)))

,metric,value
0,N videos (loaded),50
1,|V| = G (incl. background 0),19
2,#activity classes,1
3,segment length p10/50/90,"(18.0, 19.0, 23.1)"
4,published N,50
5,published #classes,1


activities: ['salad']
per-activity video counts: {'salad': 50}


## 2. Co-occurrence ground cost

In [3]:
# Data-driven co-occurrence ground cost over the symbol alphabet (symbols that frequently
# abut are closer) -> shared vocab.compute_distance_matrix. Built once; reused by the suite.
D_G = build_action_seg_ground_cost(NAME, X=X, kind='cooccurrence')
assert np.allclose(D_G, D_G.T) and np.allclose(np.diag(D_G), 0.0), 'D_G must be symmetric, zero-diagonal'
if D_G.shape != (G, G):   # loader G=len(mapping) vs ground-cost G=max(observed)+1 (a top id unused)
    print(f'NOTE: ground-cost G={D_G.shape[0]} != mapping G={G}; using {D_G.shape[0]} for shape-consistency')
    G = D_G.shape[0]
print('D_G shape:', D_G.shape, '| symmetric, zero-diagonal OK')

D_G shape: (19, 19) | symmetric, zero-diagonal OK


## 3. Generalization suite (single-group quality; order-null empty by construction)

In [4]:
res = run_generalization_suite(X, labels, G, D_G, name=NAME, upsample_to=UPSAMPLE)
res['order'].to_csv(os.path.join(OUT, 'order_null.csv'), index=False)
res['quality'].reset_index().to_csv(os.path.join(OUT, 'quality.csv'), index=False)
print(f"ORDER-NULL: {len(res['order'])} rows -- EMPTY expected (single 'salad' class -> 0 comparisons).")
print('\nQUALITY of the prototypical salad execution across averagers:')
display(res['quality'].round(3))

ORDER-NULL: 0 rows -- EMPTY expected (single 'salad' class -> 0 comparisons).

QUALITY of the prototypical salad execution across averagers:


metric,inertia_rtwe,inertia_native,freq_fidelity,struct_preservation,entropy_bits,n_distinct,n_segments,stability_inertia_rtwe,stability_histogram,dataset
method,,,,,,,,,,
dba_dtw,18.452,25.810,0.176,3.144,3.849,16.5,24.0,0.66,0.013,50salads
edit_median,13.921,19.260,0.138,2.677,3.931,16.0,17.0,0.00,0.000,50salads
k_medoid,14.055,14.055,0.097,2.072,4.062,17.0,17.0,0.00,0.000,50salads
majority_voting,18.205,0.707,0.318,2.838,3.454,13.0,21.0,0.00,0.000,50salads
soft_dtw,15.659,18.041,0.084,2.651,4.144,18.5,19.5,0.45,0.005,50salads
wasserstein,NaN,0.093,0.049,NaN,4.191,19.0,NaN,NaN,0.000,50salads


## 4. Prototypical salad execution (chronogram)

In [5]:
# Prototypical execution per activity: a deterministic edit-median barycenter of each
# activity's executions, rendered as a chronogram strip (reuses plot_cohort_barycenters).
methods = default_baseline_methods(D_G)
code_to_label = {i: n for n, i in
                 read_mapping(os.path.join(default_root(NAME), 'mapping.txt'), cfg['background']).items()}
CAP_CHRONO, rng = 60, np.random.default_rng(0)
proto = {}
for a in sorted(set(labels)):
    ia = np.where(labels == a)[0]
    if len(ia) > CAP_CHRONO:
        ia = rng.choice(ia, CAP_CHRONO, replace=False)
    Xa = np.vstack([upsample_sequence(X[i], UPSAMPLE) for i in ia]).astype(int)
    proto[a] = np.asarray(methods['edit_median']['build'](Xa, 0)).astype(int)
plot_cohort_barycenters(proto, groups=sorted(proto), code_to_label=code_to_label, mask_background=True,
                        title=f'{NAME}: prototypical execution per activity (edit-median barycenter)',
                        savepath=os.path.join(OUT, 'chronograms.png'))
print('saved', os.path.join(OUT, 'chronograms.png'))

saved /home/perochon/data-gold-final/outputs/symbolic_barycenter/generalization/50salads/chronograms.png


**Reading.** With one activity, this isolates *representation quality* from discrimination: which
averager best summarizes the 50 salad executions on each fidelity axis. CSVs: `order_null.csv`
(empty), `quality.csv`; figure `chronograms.png`.